In [ ]:
import os

OPENAI_API_KEY = os.environ('OPENAI_API_KEY')
HF_TOKEN = os.environ('HF_TOKEN')

## meta-llama/Llama-3.2-3B-Instruct
https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'meta-llama/Llama-3.2-3B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

In [ ]:
model.device

In [ ]:
prompt = 'Explain the theory of relativity in simple terms.'

inputs = tokenizer(prompt, return_tensor='pt').to(model.device)
print(inputs)

outputs = model.generate(**inputs, max_length=1024)
print(outputs)

In [ ]:
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
def generate_by_sllm(prompt: str):
    inputs = tokenizer(prompt, return_tensor='pt').to(model.device)
    outputs = model.generate(**inputs, max_length=1024)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate_by_sllm("Hello How's the weather today?"))

In [ ]:
print(generate_by_sllm('안녕 만나서 반갑다. 오늘 저녁 뭐 먹지?'))

## Bllossom/llama-3.2-Korean-Bllossom-3B

https://huggingface.co/Bllossom/llama-3.2-Korean-Bllossom-3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = 'Bllossom/llama-3.2-Korean-Bllossom-3B'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map='auto'
)

In [ ]:
instruction = "철수가 20개의 연필을 가지고 있었는데 영희가 절반을 가져가고 민수가 남은 5개를 가져갔으면 철수에게 남은 연필의 갯수는 몇개인가요?"

messages = [
    {"role": "user", "content": f"{instruction}"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

enc = {k: v.to(model.device) for k, v in input_ids.items()}

print(enc)
print(tokenizer.decode(enc['input_ids'][0], skip_special_tokens=True))

In [ ]:
terminators = [
    tokenizer.convert_tokens_to_ids("<|end_of_text|>"),
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = model.generate(
    **input_ids,
    max_new_tokens=1024,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.6,
    top_p=0.9
)

prompt_len = enc['input_ids'].shape[-1]
print(tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True))

## LGAI-EXAONE/EXAONE-4.0-1.2B
https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "LGAI-EXAONE/EXAONE-4.0-1.2B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="bfloat16",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# choose your prompt
prompt = "너가 얼마나 대단한지 설명해 봐"

messages = [
    {"role": "user", "content": prompt}
]
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

enc = {k: v.to(model.device) for k, v in input_ids.items()}

output = model.generate(
    **enc,
    max_new_tokens=128,
    do_sample=False,
)
print(tokenizer.decode(output[0]))


In [ ]:
import os
from openai import OpenAI

client = OpenAI(api_key = os.getenv('OPENAI_API_KEY'))

def generate_by_llm(prompt: str) -> str:
    response = client.response.create(
        model='gpt-5.6-luna',
        input=prompt,
        max_output_tokens=1024
    )
    return response.output_text

print(generate_by_llm('Llama-3.2-3B-Instruct, llama-3.2-Korean-Bllossom-3B, EXAONE-4.0-1.2B 차이점'))